## Data Preprocessing
### DSCI 403
#### Clara Pugh and Chandler Bergstrom

This notebook loads in satellite-based sea surface temperature data as well as ARGO float data from NOAA, converts it to a pandas dataframe, and preprocesses it so it's ready to work with. 

In [7]:
# standard imports
import numpy as np
import matplotlib.pyplot as plt

import xarray as xr
import pandas as pd

# import to load ARGO float data
from argopy import DataFetcher as ArgoDataFetcher

from dask.diagnostics import ProgressBar

import os

In [8]:
# ### Import Sea Surface Temperature (SST) data from NOAA

# # Define the OPeNDAP URL for the NOAA OISST dataset
# sst_url = "https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180"

# # Open the dataset with xarray
# ds_sst = xr.open_dataset(sst_url, chunks='auto')

# # Subset the data to specific region and time of interest, download subset to memory
# # This is Gulf Stream region in Jan 2023
# subset_sst = ds_sst.sel(time=slice('2023-01-01', '2023-01-31'), latitude=slice(25.0, 45.0), longitude=slice(-80.0, -50.0))

# # Plot a quick check with subsampled resolution to save time
# #subset_sst.analysed_sst.isel(time=0, latitude=slice(None, None, 100), longitude=slice(None, None, 100)).plot()

import time

### Import Sea Surface Temperature (SST) data from NOAA

# Define the OPeNDAP URL for the NOAA OISST dataset
sst_url = "https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180"

# Setup retry parameters
max_attempts = 3
delay_seconds = 10
ds_sst = None

# Open the dataset with xarray using a retry loop
for attempt in range(max_attempts):
    try:
        print(f"Attempt {attempt + 1} to connect to NOAA ERDDAP...")
        ds_sst = xr.open_dataset(sst_url, chunks='auto')
        print("Connection successful!")
        break
    except OSError as e:
        print(f"Connection failed: {e}")
        if attempt < max_attempts - 1:
            print(f"Retrying in {delay_seconds} seconds...")
            time.sleep(delay_seconds)
        else:
            print("All attempts failed. The server is completely offline.")
            raise  # Re-raise the exception to stop execution

# Subset the data to specific region and time of interest, download subset to memory
# This is Gulf Stream region in Jan 2023
if ds_sst is not None:
    subset_sst = ds_sst.sel(time=slice('2023-01-01', '2023-01-31'), latitude=slice(25.0, 45.0), longitude=slice(-80.0, -50.0))

Attempt 1 to connect to NOAA ERDDAP...


curl error details: 


Connection failed: [Errno -68] NetCDF: I/O failure: 'https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180'
Retrying in 10 seconds...
Attempt 2 to connect to NOAA ERDDAP...


curl error details: 


Connection failed: [Errno -68] NetCDF: I/O failure: 'https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180'
Retrying in 10 seconds...
Attempt 3 to connect to NOAA ERDDAP...
Connection failed: [Errno -68] NetCDF: I/O failure: 'https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180'
All attempts failed. The server is completely offline.


curl error details: 


OSError: [Errno -68] NetCDF: I/O failure: 'https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180'

In [ ]:
### Import ARGO Float data

# Spatial-temporal bounds format: [lon_min, lon_max, lat_min, lat_max, depth_min, depth_max, date_start, date_end]
argo_fetcher = ArgoDataFetcher().region([-80, -50, 25, 45, 0, 1000, '2023-01-01', '2023-01-31'])

# Fetch the data and convert to an xarray dataset
ds_argo = argo_fetcher.to_xarray()

# Argo data is initially loaded as a 1D list. Convert to standard depth-profile grid for easier analysis
ds_argo_profiles = ds_argo.argo.point2profile()

# Plot a quick temperature-depth profile for the first float
ds_argo_profiles.isel(N_PROF=0).plot.scatter(x="TEMP", y="PRES", yincrease=False, edgecolors='none')

In [ ]:
#Matching cell
# Calculate the spatial and temporal bounding box of Argo profiles
min_lon, max_lon = ds_argo_profiles['LONGITUDE'].min().values, ds_argo_profiles['LONGITUDE'].max().values
min_lat, max_lat = ds_argo_profiles['LATITUDE'].min().values, ds_argo_profiles['LATITUDE'].max().values
min_time, max_time = ds_argo_profiles['TIME'].min().values, ds_argo_profiles['TIME'].max().values

# Slice the spatial bounding box, then subsample by taking every 10th pixel
regional_sst = ds_sst.sel(time=slice(min_time, max_time), latitude=slice(min_lat, max_lat), longitude=slice(min_lon, max_lon)).isel(
    latitude=slice(None, None, 10), longitude=slice(None, None, 10))

In [ ]:
# download cell
# Download regional block into local memory
from dask.diagnostics import ProgressBar

with ProgressBar():
    regional_sst_loaded = regional_sst.compute()

In [ ]:
# get SST data from the ARGO locations
colocated_sst = regional_sst_loaded.sst.sel(time=ds_argo_profiles['TIME'], latitude=ds_argo_profiles['LATITUDE'],
                                                     longitude=ds_argo_profiles['LONGITUDE'], method='nearest')

# Add colocated SST back into Argo dataset as a new variable
ds_argo_profiles['matched_sst'] = colocated_sst

# Convert the matched Argo/SST data into a pandas dataframe (GS_0123 means gulf stream january 2023)
df_GS_0123 = ds_argo_profiles.to_dataframe()

df_GS_0123.head()

In [ ]:
df_GS_0123.info()

- There a null values that we will have to deal with in the columns: PRES, PRES_ERROR, PSAL, PSAL_ERROR, TEMP, TEMP_ERROR
- There are a few non-numeric columns: DATA_MODE and DIRECTION are strings, and TIME and tim are in datetime64

### Chandler input below

I made a copy of Clara's work and am continuing it below!

In [ ]:
# calcs the euclidean distance

df = df_GS_0123.reset_index()
df["match_dist_deg"] = np.hypot(df["LATITUDE"] - df["latitude"],
                                df["LONGITUDE"] - df["longitude"])
print(df["match_dist_deg"].describe())

In [ ]:
print(df["TEMP_QC"].value_counts())
print(df["DATA_MODE"].value_counts())

In [ ]:
# filtering bad data, calc temp outliers, printing stats

n0 = len(df)
df = df[df["TEMP_QC"].isin([1, 2]) & df["PRES_QC"].isin([1, 2])]
df = df.dropna(subset=["TEMP", "PRES"])
print(n0, "->", len(df), "rows")
df = df.drop(columns=["time", "latitude", "longitude"])  # redundant after match distance
df["TEMP_MINUS_SST"] = df["TEMP"] - df["matched_sst"]
surface = df[df["PRES"] <= 10]
print(surface["TEMP_MINUS_SST"].describe())

### Match distance in km, cleans the data, and compares by mode

In [ ]:
# calc distances and filtering more data

df = df_GS_0123.reset_index()
dy = (df["LATITUDE"] - df["latitude"]) * 111
dx = (df["LONGITUDE"] - df["longitude"]) * 111 * np.cos(np.radians(df["LATITUDE"]))
df["match_dist_km"] = np.hypot(dx, dy)
df = df.dropna(subset=["TEMP", "PRES"])
df["TEMP_MINUS_SST"] = df["TEMP"] - df["matched_sst"]
surface = df[df["PRES"] <= 10]
print(len(df), len(surface))
print(surface[["match_dist_km", "TEMP_MINUS_SST"]].describe())
print(surface.groupby("DATA_MODE")["TEMP_MINUS_SST"].describe())

### Meaning:

What the numbers say

Median difference is about 0.09 °C, and the middle 50% of rows are between about -0.4 and +0.5 °C. 
Tails are extreme- the std (2.1 °C) is a lot bigger than middle sections spread, and the range is -14.6 to 9.4 °C. 
* A 14 °C gap between two measurements of the same ocean surface isn’t a measurement difference. It’s either a mismatch (the satellite pixel was up to 190 km away, possibly across the Gulf Stream front, where there are sharp temperature cliffs) or a bad value.

  
D, R and A all are similar (~2 °C). 
Only 1,741 of the 97,145 valid rows are near-surface (≤ 10 dbar, about 8 per profile).

In [ ]:
surface = surface.copy()
surface["ABS_DIFF"] = surface["TEMP_MINUS_SST"].abs()

# does the error grow with match distance?
surface["dist_bin"] = pd.cut(surface["match_dist_km"], [0, 50, 100, 150, 200])
print(surface.groupby("dist_bin", observed=True)["ABS_DIFF"].agg(["count", "median", "std"]))

# look at the worst cases
cols = ["PLATFORM_NUMBER", "CYCLE_NUMBER", "TIME", "LATITUDE", "LONGITUDE",
        "PRES", "TEMP", "matched_sst", "match_dist_km", "TEMP_MINUS_SST"]
print(surface.sort_values("ABS_DIFF", ascending=False)[cols].head(15))

In [ ]:
# w/ out pandas 
surface = surface.copy()
surface["ABS_DIFF"] = surface["TEMP_MINUS_SST"].abs()

edges = [0, 50, 100, 150, 200]
bins = np.digitize(surface["match_dist_km"], edges)
for b in range(1, len(edges)):
    sel = surface.loc[bins == b, "ABS_DIFF"]
    print(f"{edges[b-1]}-{edges[b]} km: n={len(sel)}, median={sel.median():.2f}, std={sel.std():.2f}")

cols = ["PLATFORM_NUMBER", "CYCLE_NUMBER", "TIME", "LATITUDE", "LONGITUDE",
        "PRES", "TEMP", "matched_sst", "match_dist_km", "TEMP_MINUS_SST"]
print(surface.sort_values("ABS_DIFF", ascending=False)[cols].head(15))

In [ ]:
df_GS_0123.to_pickle("df_GS_0123.pkl")

In [ ]:
df_GS_0123 = pd.read_pickle("df_GS_0123.pkl")

In [ ]:
print(os.getcwd())

In [ ]:

"""
Preprocessing pipeline for the matched Argo + OISST dataframe.
 
Usage in the notebook (paste the function into a cell, then):
    clean, surface, prof = preprocess(df_GS_0123)          # or df_GS_0123_full
 
Column names assumed (from the notebook): PLATFORM_NUMBER, CYCLE_NUMBER, TIME,
LATITUDE, LONGITUDE, PRES, TEMP, PSAL, DATA_MODE, matched_sst, and the matched
satellite grid point in lowercase latitude / longitude.
"""
import numpy as np
import pandas as pd
 
LOG = []  # keeps a record of every step for the write-up
 
 
def log(step, n_before, n_after):
    LOG.append({"step": step, "rows_before": n_before, "rows_after": n_after,
                "removed": n_before - n_after})
    print(f"{step}: {n_before} -> {n_after} ({n_before - n_after} removed)")
 
 
def preprocess(df_in, max_match_km=None, surface_pres=10.0):
    LOG.clear()
 
    # Flattening the (profile, level) multi index into ordinary columns
    df = df_in.reset_index()
    n = len(df)
    print("Starting rows:", n)
 
    # Match distance in km between each float and matched satellite pixel
    dy = (df["LATITUDE"] - df["latitude"]) * 111.0
    dx = (df["LONGITUDE"] - df["longitude"]) * 111.0 * np.cos(np.radians(df["LATITUDE"]))
    df["match_dist_km"] = np.hypot(dx, dy)
 
    # remove padding rows 
    df = df.dropna(subset=["TEMP", "PRES"])
    log("Removed padding / missing TEMP or PRES", n, len(df)); n = len(df)
 
    # Remove physically impossible values
    df = df[df["TEMP"].between(-2.5, 40) & (df["PRES"] >= 0)]
    df = df[df["matched_sst"].between(-2.5, 40)]
    log("Removed out-of-range values", n, len(df)); n = len(df)
 
    # Remove duplicate rows
    df = df.drop_duplicates(subset=["PLATFORM_NUMBER", "CYCLE_NUMBER", "PRES", "TEMP"])
    log("Removed duplicates", n, len(df)); n = len(df)
 
    # Drop redundant columns 
    drop_cols = ["time", "latitude", "longitude", "POSITION_QC", "TIME_QC",
                 "PRES_QC", "TEMP_QC", "PSAL_QC"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
 
    # Derived variables
    df["TEMP_MINUS_SST"] = df["TEMP"] - df["matched_sst"]
    df["MONTH_DAY"] = pd.to_datetime(df["TIME"]).dt.strftime("%m-%d")
 
    # Near-surface subset
    surface = df[df["PRES"] <= surface_pres].copy()
    log(f"Near-surface subset (PRES <= {surface_pres} dbar)", len(df), len(surface))
 
    # limit to well-matched pixels
    if max_match_km is not None:
        before = len(surface)
        surface = surface[surface["match_dist_km"] <= max_match_km]
        log(f"Kept matches within {max_match_km} km", before, len(surface))
 
    # Collapse to one row per profile 
    prof = (surface.groupby(["PLATFORM_NUMBER", "CYCLE_NUMBER"])
            .agg(TIME=("TIME", "first"), LAT=("LATITUDE", "first"),
                 LON=("LONGITUDE", "first"), DATA_MODE=("DATA_MODE", "first"),
                 TEMP_SURF=("TEMP", "median"), SST=("matched_sst", "first"),
                 DIST_KM=("match_dist_km", "first"), N_OBS=("TEMP", "size"))
            .reset_index())
    prof["DIFF"] = prof["TEMP_SURF"] - prof["SST"]
 
    # Flag outliers with robust rule (median / MAD), *dont delete* 
    med = prof["DIFF"].median()
    mad = (prof["DIFF"] - med).abs().median() * 1.4826  # MAD scaled to ~SD
    prof["OUTLIER"] = (prof["DIFF"] - med).abs() > 3 * mad
    print(f"\nProfiles: {len(prof)}  | flagged outliers: {prof['OUTLIER'].sum()}")
    print(f"Median diff: {med:.2f} C, robust SD: {mad:.2f} C")
 
    return df.reset_index(drop=True), surface.reset_index(drop=True), prof
 
 
if __name__ == "__main__":
    pass

In [ ]:
clean, surface, prof = preprocess(df_GS_0123)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
 
 
def plot_summary(prof, save_as="argo_sst_summary.png"):
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
 
    # Argo near surface temp vs staellite SST, by distance
    ax = axes[0]
    sc = ax.scatter(prof["SST"], prof["TEMP_SURF"], c=prof["DIST_KM"],
                    cmap="viridis", s=25, edgecolor="none")
    lo = min(prof["SST"].min(), prof["TEMP_SURF"].min()) - 0.5
    hi = max(prof["SST"].max(), prof["TEMP_SURF"].max()) + 0.5
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="1:1 line")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel("Satellite SST (°C)")
    ax.set_ylabel("Argo near-surface temperature (°C)")
    ax.set_title("Argo vs satellite SST")
    ax.legend(loc="upper left")
    fig.colorbar(sc, ax=ax, label="Match distance (km)")
 
    # map of profile locations, colored by temp difference
    ax = axes[1]
    lim = np.nanpercentile(np.abs(prof["DIFF"]), 95)
    sc = ax.scatter(prof["LON"], prof["LAT"], c=prof["DIFF"], cmap="RdBu_r",
                    vmin=-lim, vmax=lim, s=35, edgecolor="none")
    out = prof[prof["OUTLIER"]]
    ax.scatter(out["LON"], out["LAT"], facecolors="none", edgecolors="k",
               s=70, label="Flagged outlier")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title("Argo minus satellite SST by location")
    ax.legend(loc="lower right")
    fig.colorbar(sc, ax=ax, label="Difference (°C)")
 
    # histogram of the differences
    ax = axes[2]
    ax.hist(prof["DIFF"], bins=40, color="steelblue", edgecolor="white")
    ax.axvline(prof["DIFF"].median(), color="k", ls="--",
               label=f"Median = {prof['DIFF'].median():.2f} °C")
    ax.set_xlabel("Argo minus satellite SST (°C)")
    ax.set_ylabel("Number of profiles")
    ax.set_title("Distribution of differences")
    ax.legend()
 
    plt.tight_layout()
    plt.savefig(save_as, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
plot_summary(prof)